In [ ]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
from uuid import uuid4
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import OpenAIEmbeddings

C:\Users\admin\AppData\Local\Temp\ipykernel_17516\57761500.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [ ]:

from pymongo import MongoClient
from pymongo.server_api import ServerApi

uri = "mongodb+srv://:@cluster0.nnjobcv.mongodb.net/?appName=Cluster0"

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [3]:
load_dotenv()
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [4]:
DATA_DIR = Path.cwd().resolve()
if not (DATA_DIR / "data").exists():
    DATA_DIR = DATA_DIR.parent

DATA_DIR = (DATA_DIR / "data").resolve()

In [5]:
pdf_files_paths = DATA_DIR.glob("*.pdf")

In [6]:
list(pdf_files_paths)

[WindowsPath('D:/2027/Projects/RAG/data/Atomic habits ( PDFDrive ).pdf'),
 WindowsPath('D:/2027/Projects/RAG/data/attention.pdf'),
 WindowsPath('D:/2027/Projects/RAG/data/BhagavadGita.pdf'),
 WindowsPath('D:/2027/Projects/RAG/data/the-5-am-club.pdf')]

In [7]:
def normalize_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text.lower()).strip("_")


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()

In [8]:
all_documents = []
mongo_documents = []
source_ids = set()

for pdf_file_path in list(DATA_DIR.glob("*.pdf"))[:1]:
    print(f"{pdf_file_path.name}: {file_sha256(pdf_file_path)}")
    loader = PyMuPDFLoader(str(pdf_file_path))
    documents = loader.load()

    source_checksum = file_sha256(pdf_file_path)
    source_id = normalize_name(pdf_file_path.stem)
    source_ids.add(source_id)

    for index, doc in enumerate(documents):
        chunk_text = (doc.page_content or "").strip()
        if not chunk_text:
            continue

        metadata = {
            "source": pdf_file_path.name,
            "source_id": source_id,
            "source_checksum": source_checksum,
            "page_number": index + 1,
            "chunk_id": str(uuid4()),
            "chunk_index": index,
            "chunk_size": len(chunk_text),
            "created_at": datetime.now(timezone.utc).isoformat(),
        }

        mongo_documents.append({
            "_id": metadata["chunk_id"],
            "text": chunk_text,
            "embedding": [],
            "metadata": metadata,
        })

        doc.metadata.update(metadata)
        all_documents.append(doc)

print(len(all_documents), "documents loaded from all PDFs.")
print(len(mongo_documents), "mongo documents prepared for hybrid retrieval.")
sorted(set(source_ids))


Atomic habits ( PDFDrive ).pdf: a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878
254 documents loaded from all PDFs.
254 mongo documents prepared for hybrid retrieval.


['atomic_habits___pdfdrive']

In [9]:
from langchain_experimental.text_splitter import SemanticChunker

C:\Users\admin\AppData\Local\Temp\ipykernel_6208\2829801429.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [10]:
load_dotenv()

True

In [10]:
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [12]:
semantic_chunker = SemanticChunker(
        embeddings=embeddings
    )

In [14]:
# {
#     "_id": "doc-001",
#     "text": "...",
#     "embedding": [0.12, 0.45, 0.78, ...],
#     "metadata": {
#         "source": "attention.pdf",
#         "page_number": 2,
#         "category": "research"
#     }
# }

In [13]:
mongo_documents[0]

{'_id': '8a0e5a8f-0a17-4931-a800-ad7d3116d3f4',
 'text': 'AN IMPRINT OF PENGUIN RANDOM HOUSE LLC\n375 Hudson Street\nNew York, New York 10014\nCopyright © 2018 by James Clear\nPenguin supports copyright. Copyright fuels creativity, encourages diverse voices, promotes free speech, and creates a vibrant culture. Thank you for buying an authorized edition of this book and for\ncomplying with copyright laws by not reproducing, scanning, or distributing any part of it in any form without permission. You are supporting writers and allowing Penguin to continue to publish books\nfor every reader.',
 'embedding': [],
 'metadata': {'source': 'Atomic habits ( PDFDrive ).pdf',
  'source_id': 'atomic_habits___pdfdrive',
  'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878',
  'page_number': 3,
  'chunk_id': '8a0e5a8f-0a17-4931-a800-ad7d3116d3f4',
  'chunk_index': 2,
  'chunk_size': 531,
  'created_at': '2026-08-04T23:03:07.142232+00:00'}}

In [16]:
BATCH_SIZE = 100

for start_index in range(0, len(mongo_documents), BATCH_SIZE):
    end_index = min(start_index + BATCH_SIZE, len(mongo_documents))
    print(f"Processing documents {start_index + 1}-{end_index}/{len(mongo_documents)}")

    batch_documents = mongo_documents[start_index:end_index]
    batch_texts = [doc["text"] for doc in batch_documents]
    batch_embeddings = embeddings.embed_documents(batch_texts)

    for doc, embedding in zip(batch_documents, batch_embeddings):
        doc["embedding"] = embedding

Processing documents 1-100/254
Processing documents 101-200/254
Processing documents 201-254/254


In [18]:
mongo_documents[0].keys()

dict_keys(['_id', 'text', 'embedding', 'metadata'])

In [ ]:
DB_NAME = "RAG_2027"
COLLECTION_NAME = "book_store"

db = client[DB_NAME]
collection = db[COLLECTION_NAME]


In [ ]:
# operations = [
#     ReplaceOne({"_id": doc["_id"]}, doc, upsert=True)
#     for doc in mongo_documents
# ]
# result = collection.bulk_write(operations)
# print(f"Upserted: {result.upserted_count}, Modified: {result.modified_count}, Total: {len(mongo_documents)}")


Upserted: 254, Modified: 0, Total: 254


# Create vector search index

In [23]:
from pymongo.operations import SearchIndexModel

In [24]:
VECTOR_SEARCH_INDEX_NAME = "vector_index"

In [ ]:

embedding_dim = len(embeddings.embed_query("Hello"))

vector_index = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": embedding_dim,
                "similarity": "cosine",
            }
        ]
    },
    name=VECTOR_SEARCH_INDEX_NAME,
    type="vectorSearch",
)

print(f"Creating Atlas vector search index '{VECTOR_SEARCH_INDEX_NAME}'...")


Creating Atlas vector search index 'vector_index'...


In [17]:
result = collection.create_search_indexes([vector_index])
print(result)


['vector_index']


In [ ]:
from pymongo.operations import SearchIndexModel

SPARSE_INDEX_NAME = "sparse_index"

sparse_index = SearchIndexModel(
    definition={
        "mappings": {
            "dynamic": False,
            "fields": {
                "text": {
                    "type": "string",
                    "analyzer": "lucene.standard"
                }
            }
        }
    },
    name=SPARSE_INDEX_NAME,
    type="search"
)

print(f"Creating sparse search index '{SPARSE_INDEX_NAME}'...")
result = collection.create_search_indexes([sparse_index])
print(result)

In [18]:
# Create Essembled retriver with both vector and sparse search indexes

Creating sparse search index 'sparse_index'...
['sparse_index']


### Create dense retriever with vector search index

In [25]:
from langchain_mongodb import MongoDBAtlasVectorSearch

vector_store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=embeddings,
    index_name=VECTOR_SEARCH_INDEX_NAME,
    text_key="text",
    embedding_key="embedding",
)

dense_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Dense retriever ready.")

Dense retriever ready.


In [26]:
dense_retriever.invoke(
    input="what does Enterpriner do for living",
    k=5
)

[Document(id='5f4d7bcc-1adb-4d96-995d-ab0eba69445d', metadata={'_id': '5f4d7bcc-1adb-4d96-995d-ab0eba69445d', 'metadata': {'source': 'Atomic habits ( PDFDrive ).pdf', 'source_id': 'atomic_habits___pdfdrive', 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878', 'page_number': 124, 'chunk_id': '5f4d7bcc-1adb-4d96-995d-ab0eba69445d', 'chunk_index': 123, 'chunk_size': 2389, 'created_at': '2026-08-04T23:03:07.144227+00:00'}}, page_content='PRIME THE ENVIRONMENT FOR FUTURE USE\nOswald Nuckols is an IT developer from Natchez, Mississippi. He is also\nsomeone who understands the power of priming his environment.\nNuckols dialed in his cleaning habits by following a strategy he refers to as\n“resetting the room.” For instance, when he finishes watching television, he\nplaces the remote back on the TV stand, arranges the pillows on the couch, and\nfolds the blanket. When he leaves his car, he throws any trash away. Whenever\nhe takes a shower, he wipes down the 

### Create a full-text retriever using the Atlas search index

In [2]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from langchain_mongodb.retrievers.full_text_search import MongoDBAtlasFullTextSearchRetriever

uri = "mongodb+srv://:@cluster0.nnjobcv.mongodb.net/?appName=Cluster0"
client = MongoClient(uri, server_api=ServerApi('1'))
DB_NAME = "RAG_2027"
COLLECTION_NAME = "book_store"
SPARSE_INDEX_NAME = "sparse_index"
collection = client[DB_NAME][COLLECTION_NAME]

full_text_retriever = MongoDBAtlasFullTextSearchRetriever(
    collection=collection,
    search_field="text",
    search_index_name=SPARSE_INDEX_NAME,
)

query = "What is enterprener name"
full_text_docs = full_text_retriever.invoke(query)
print("Full-text retriever ready.")
for i, doc in enumerate(full_text_docs[:3], 1):
    print(f"Result {i}")
    print(doc.page_content[:300])
    print("---")

NameError: name 'collection' is not defined

## Step 1: Add Reciprocal Rank Fusion (RRF)
RRF combines the dense retriever and full-text retriever rankings into one fused ranking before the optional cross-encoder rerank.

In [ ]:
from langchain_core.documents import Document


def _document_key(document: Document) -> str:
    metadata = document.metadata or {}
    nested_metadata = metadata.get("metadata", {}) if isinstance(metadata.get("metadata"), dict) else {}
    return str(
        metadata.get("chunk_id")
        or nested_metadata.get("chunk_id")
        or metadata.get("_id")
        or nested_metadata.get("_id")
        or f"{nested_metadata.get('source', metadata.get('source', 'unknown'))}:{nested_metadata.get('page_number', metadata.get('page_number', 'n/a'))}:{document.page_content[:160]}"
    )


def reciprocal_rank_fusion(retriever_results, rrf_k: int = 60, top_n: int = 10):
    fused_scores = {}
    document_lookup = {}

    for documents in retriever_results:
        for rank, document in enumerate(documents, start=1):
            key = _document_key(document)
            document_lookup.setdefault(key, document)
            fused_scores[key] = fused_scores.get(key, 0.0) + 1.0 / (rank + rrf_k)

    ranked_documents = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)[:top_n]

    fused_documents = []
    for fused_rank, (key, score) in enumerate(ranked_documents, start=1):
        source_document = document_lookup[key]
        metadata = dict(source_document.metadata or {})
        metadata["rrf_score"] = score
        metadata["rrf_rank"] = fused_rank
        fused_documents.append(
            Document(
                page_content=source_document.page_content,
                metadata=metadata,
            )
        )

    return fused_documents


query = "summarize the habit stacking rule"
dense_docs = dense_retriever.invoke(query)
full_text_docs = full_text_retriever.invoke(query)

hybrid_results = reciprocal_rank_fusion([dense_docs, full_text_docs], rrf_k=60, top_n=10)
rrf_results = hybrid_results

print(f"Dense candidates: {len(dense_docs)}")
print(f"Full-text candidates: {len(full_text_docs)}")
print(f"RRF fused results: {len(hybrid_results)}")
for i, doc in enumerate(hybrid_results, 1):
    metadata = doc.metadata or {}
    nested_metadata = metadata.get("metadata", {}) if isinstance(metadata.get("metadata"), dict) else {}
    source = nested_metadata.get("source", metadata.get("source", "unknown"))
    page_number = nested_metadata.get("page_number", metadata.get("page_number", "n/a"))
    print(f"\nResult {i}")
    print("RRF Score:", doc.metadata.get("rrf_score"))
    print("Source:", source)
    print("Page:", page_number)
    print(doc.page_content[:250])

Hybrid results: 30

Result 1
Source: atomic_habit.pdf
Page: 64
FIGURE 7: Habit stacking increases the likelihood that you’ll stick with a habit by stacking your new behavior on top of an old one. This process can be repeated to chain numerous habits
together, each one acting as the cue for the next. Your morning

Result 2
Source: Atomic habits ( PDFDrive ).pdf
Page: 64
FIGURE 7: Habit stacking increases the likelihood that you’ll stick with a habit by stacking your new behavior on top of an old one. This process can be repeated to chain numerous habits
together, each one acting as the cue for the next.
Your morning

Result 3
Source: atomic_habit.pdf
Page: 62
Each action becomes a cue that triggers
the next behavior. Why is this important? When it comes to building new habits, you can use the connectedness of
behavior to your advantage. One of the best ways to build a new habit is to
identify a current ha

Result 4
Source: Atomic habits ( PDFDrive ).pdf
Page: 62
acquired a scarlet robe 

## Step 2: Optional cross-encoder reranking after RRF
Use the fused RRF results as the candidate set for the cross-encoder reranker.

In [31]:
from sentence_transformers import CrossEncoder

In [32]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

d:\2027\Projects\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [58]:
hybrid_results[0]

Document(metadata={'_id': 'a26ccbd0-f25c-4d9f-a8bd-801ac1c9c9f0', 'metadata': {'source': 'Atomic habits ( PDFDrive ).pdf', 'source_id': 'atomic_habits___pdfdrive', 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878', 'page_number': 208, 'chunk_id': 'a26ccbd0-f25c-4d9f-a8bd-801ac1c9c9f0', 'chunk_index': 207, 'chunk_size': 780, 'created_at': '2026-08-04T23:03:07.147231+00:00'}, 'vector_score': 0.011475409836065573, 'rank': 11, 'score': 0.01564207650273224, 'fulltext_score': 0.004166666666666667}, page_content='INTRODUCTION\nWe all deal with setbacks: What about luck, you might ask? Luck matters, certainly. Habits are not the only thing that influence your success, but they are probably the most important factor that is\nwithin your control. And the only selfimprovement strategy that makes any sense is to focus on what you can control.\nThe entrepreneur and investor Naval Ravikant: Naval Ravikant (@naval), “To write a great book, you must first become the

In [60]:
pairs = []
for doc in hybrid_results:
    pairs.append([query, doc.page_content])

In [61]:

scores = cross_encoder.predict(pairs)
scores
     

array([  5.4041023 ,   5.4041023 ,   4.452841  ,   1.1032655 ,
         4.2464857 ,   3.2173727 ,   3.2173727 ,   4.2464857 ,
         0.5919764 ,   1.200598  ,   0.70370525,   0.70370525,
         1.200598  ,   1.3082598 ,   1.3082598 ,   2.2883942 ,
         4.525485  ,   4.334015  ,   4.103912  ,  -0.20499793,
        -0.20499793,  -0.86095333,   0.79029727,  -0.86095333,
        -4.742342  , -10.978537  ,   0.01958617,  -6.623334  ,
        -3.333003  ,  -4.5581203 ], dtype=float32)

In [66]:
scored_docs = [
    {
        "score": float(score),
        "page_content": doc.page_content,
        "metadata": doc.metadata,
    }
    for score, doc in zip(scores, hybrid_results)
]

scored_docs[:3]

[{'score': 5.404102325439453,
  'page_content': 'FIGURE 7: Habit stacking increases the likelihood that you’ll stick with a habit by stacking your new behavior on top of an old one. This process can be repeated to chain numerous habits\ntogether, each one acting as the cue for the next. Your morning routine habit stack might look like this:\n1. After I pour my morning cup of coffee, I will meditate for sixty\nseconds. 2. After I meditate for sixty seconds, I will write my to-do list for the\nday. 3. After I write my to-do list for the day, I will immediately begin my\nfirst task. Or, consider this habit stack in the evening:',
  'metadata': {'_id': '6a7286748e852d865eb8ee5a',
   'metadata': {'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
    'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
    'creationdate': '2020-04-30T18:46:22+00:00',
    'source': 'atomic_habit.pdf',
    'file_path': 'D:\\2027\\Projects\\RAG\\data\\atomic_habit.pdf',
    'total_pages': 256,
    '

In [68]:
reranked_document_cross_encoder = sorted(
    scored_docs,
    key=lambda item: item["score"],
    reverse=True,
)

TOP_K_FOR_LLM = 5
final_llm_docs = reranked_document_cross_encoder[:TOP_K_FOR_LLM]

final_llm_context_blocks = []
for i, item in enumerate(final_llm_docs, 1):
    metadata = item.get("metadata", {}) or {}
    nested_metadata = metadata.get("metadata", {}) if isinstance(metadata, dict) else {}
    source = nested_metadata.get("source", metadata.get("source", "unknown"))
    page_number = nested_metadata.get("page_number", metadata.get("page_number", "n/a"))

    final_llm_context_blocks.append(
        (
            f"[Rank: {i}] [Score: {item['score']:.4f}] "
            f"[Source: {source}] [Page: {page_number}]\n"
            f"{item['page_content']}"
        )
    )

final_llm_context = "\n\n".join(final_llm_context_blocks)

print(f"Reranked docs: {len(reranked_document_cross_encoder)}")
print(f"Final docs for LLM context: {len(final_llm_docs)}")
final_llm_context[:1500]

Reranked docs: 30
Final docs for LLM context: 5


'[Rank: 1] [Score: 5.4041] [Source: atomic_habit.pdf] [Page: 64]\nFIGURE 7: Habit stacking increases the likelihood that you’ll stick with a habit by stacking your new behavior on top of an old one. This process can be repeated to chain numerous habits\ntogether, each one acting as the cue for the next. Your morning routine habit stack might look like this:\n1. After I pour my morning cup of coffee, I will meditate for sixty\nseconds. 2. After I meditate for sixty seconds, I will write my to-do list for the\nday. 3. After I write my to-do list for the day, I will immediately begin my\nfirst task. Or, consider this habit stack in the evening:\n\n[Rank: 2] [Score: 5.4041] [Source: Atomic habits ( PDFDrive ).pdf] [Page: 64]\nFIGURE 7: Habit stacking increases the likelihood that you’ll stick with a habit by stacking your new behavior on top of an old one. This process can be repeated to chain numerous habits\ntogether, each one acting as the cue for the next.\nYour morning routine habit s